# Baseline Model
Baseline model for ufc predictions. Uses fighter differentials as features for simple stats like slpm, str_acc, etc

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
from datetime import datetime
import wandb
import joblib
import warnings
pd.set_option("display.max_columns", None)
warnings.filterwarnings(
    "ignore",
    message="Could not find the number of physical cores*"
)

In [2]:
wandb.login()
df = pd.read_csv('../datasets/v1.csv')
df.head()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\marvi\_netrc.
wandb: Currently logged in as: marvin3742 (marvin3742-university-of-south-florida) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


,fight_id,event_name,fight_date,weight_class,title_bout,winner,win_method,f1_fighter_id,f1_name,f1_height,f1_weight,f1_reach,f1_stance,f1_dob,f1_wins,f1_losses,f1_draws,f1_slpm,f1_str_acc,f1_sapm,f1_str_def,f1_td_avg,f1_td_acc,f1_td_def,f1_sub_avg,f2_fighter_id,f2_name,f2_height,f2_weight,f2_reach,f2_stance,f2_dob,f2_wins,f2_losses,f2_draws,f2_slpm,f2_str_acc,f2_sapm,f2_str_def,f2_td_avg,f2_td_acc,f2_td_def,f2_sub_avg
0,1,UFC 2: No Way Out,1994-03-11,Open Weight Bout,False,Scott Morris,Submission,874,Sean Daugherty,72,175,0,NaN,1975-12-04,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2715,Scott Morris,70,210,0,Orthodox,1900-01-01,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,UFC 2: No Way Out,1994-03-11,Open Weight Bout,False,Patrick Smith,Submission,3747,Patrick Smith,74,225,0,Orthodox,1963-08-28,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4330,Ray Wizard,0,0,0,NaN,1900-01-01,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,UFC 2: No Way Out,1994-03-11,Open Weight Bout,False,Johnny Rhodes,KO/TKO,2233,David Levicki,77,275,0,NaN,1900-01-01,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3292,Johnny Rhodes,72,210,0,Orthodox,1900-01-01,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,UFC 2: No Way Out,1994-03-11,Open Weight Bout,False,Frank Hamaker,Submission,1558,Frank Hamaker,0,0,0,NaN,1900-01-01,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2328,Thaddeus Luster,75,210,0,NaN,1900-01-01,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,UFC 2: No Way Out,1994-03-11,Open Weight Bout,False,Orlando Wiet,KO/TKO,2314,Robert Lucarelli,74,245,0,NaN,1900-01-01,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4295,Orlando Wiet,70,170,0,Southpaw,1965-10-24,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df = df.dropna()
df = df[df["f1_dob"] != "1900-01-01"]
df = df[df["f1_reach"] > 0]
df = df[df["f2_reach"] > 0]
df = df[df["f1_stance"] != "Open Stance"]
df = df[df["f2_stance"] != "Open Stance"]

columns = ["_height", "_weight", "_reach"]
for col in columns:
    print(f'Times {col} = 0 :{len(df[df['f1' + col] <= 0])}')

Times _height = 0 :0
Times _weight = 0 :0
Times _reach = 0 :0


In [5]:
df.shape

(4565, 43)

In [14]:
def is_winner(winner, f1):
    if winner == f1:
        return 1
    return 0

def date_diff(date_1, date_2):
    d1 = datetime.strptime(date_1, "%Y-%m-%d")
    d2 = datetime.strptime(date_2, "%Y-%m-%d")
    diff = d1-d2
    return diff.days/365

df["title_bout"] = df["title_bout"].astype(int)
df["winner"] = df.apply(lambda row: is_winner(row["winner"], row["f1_name"]), axis=1)

In [15]:
prefix = ['f1', 'f2']
features = ["_slpm", "_str_acc", "_sapm", "_str_def", "_td_avg", "_td_acc", "_td_def", "_sub_avg"]

# for pre in prefix:
#     for feature in features:
#         feature = pre + feature
#         data = df[feature]
#         plt.hist(data, bins=50)
#         plt.xlabel(feature)
#         plt.ylabel("frequency")
#         plt.title(f'Distribution of {feature}')
#         plt.show()

In [16]:
train_df, test_df = train_test_split(df, test_size=0.15, shuffle=False) # Is this technically not time series data? 
x_train, y_train, x_test, y_test = train_df.drop(["winner", "fight_id", "event_name", "fight_date", "weight_class", "win_method", "f1_fighter_id", 'f1_name', "f2_fighter_id", "f2_name"], axis=1).copy(), train_df["winner"].copy(), test_df.drop(["winner", "fight_date", "fight_id", "event_name", "weight_class", "win_method", "f1_fighter_id", 'f1_name', "f2_fighter_id", "f2_name"], axis=1).copy(), test_df["winner"].copy(),

In [17]:
diff_features = ["_slpm", "_str_acc", "_sapm", "_str_def", "_td_avg", "_td_acc", "_td_def", "_sub_avg", "_height", "_weight", "_reach", "_wins", "_losses", "_draws"]
for feature in diff_features:
    x_train["diff" + feature] = x_train["f1" + feature] - x_train["f2" + feature]
    x_train.drop(columns=["f1"+ feature, "f2" + feature], axis=1, inplace=True)
    x_test["diff" + feature] = x_test["f1" + feature] - x_test["f2" + feature]
    x_test.drop(columns=["f1"+ feature, "f2" + feature], axis=1, inplace=True)

In [18]:
x_train["diff_dob"] = x_train.apply(lambda row: date_diff(row["f1_dob"], row["f2_dob"]), axis=1)
x_test["diff_dob"] = x_test.apply(lambda row: date_diff(row["f1_dob"], row["f2_dob"]), axis=1)
x_train.drop(columns=["f1_dob", "f2_dob"], axis=1, inplace=True)
x_test.drop(columns=["f1_dob", "f2_dob"], axis=1, inplace=True)

x_train["stance_pair"] = x_train["f1_stance"] + "_vs_" + x_train["f2_stance"]
dummies = pd.get_dummies(x_train["stance_pair"], dtype=int)
x_train = pd.concat([x_train.drop(columns=["stance_pair", "f1_stance", "f2_stance"]), dummies], axis=1)

x_test["stance_pair"] = x_test["f1_stance"] + "_vs_" + x_test["f2_stance"]
dummies = pd.get_dummies(x_test["stance_pair"], dtype=int)
x_test = pd.concat([x_test.drop(columns=["stance_pair", "f1_stance", "f2_stance"]), dummies], axis=1)

In [19]:
x_train.head()

,title_bout,diff_slpm,diff_str_acc,diff_sapm,diff_str_def,diff_td_avg,diff_td_acc,diff_td_def,diff_sub_avg,diff_height,diff_weight,diff_reach,diff_wins,diff_losses,diff_draws,diff_dob,Orthodox_vs_Orthodox,Orthodox_vs_Southpaw,Orthodox_vs_Switch,Southpaw_vs_Orthodox,Southpaw_vs_Southpaw,Southpaw_vs_Switch,Switch_vs_Orthodox,Switch_vs_Southpaw,Switch_vs_Switch
191,0,-2.119048,0.097279,2.174370,-0.422249,5.063025,-0.500000,0.000000,0.000000,-1,-20,-2,-1,2,0,5.690411,1,0,0,0,0,0,0,0,0
260,0,-0.553676,-0.026667,0.289002,-0.035714,-0.424088,0.750000,-0.307692,0.000000,5,-42,2,1,1,0,-1.087671,1,0,0,0,0,0,0,0,0
274,1,-2.794408,-0.241437,0.056114,-0.142857,2.577663,-0.225806,1.000000,-4.271029,3,20,0,2,2,0,3.950685,1,0,0,0,0,0,0,0,0
298,1,-1.515392,0.096214,-0.740699,0.088889,-2.524079,0.212121,0.000000,-0.424929,-3,0,0,-6,-2,0,-4.054795,1,0,0,0,0,0,0,0,0
314,1,1.994116,0.322542,-0.984539,0.094905,5.202076,0.000000,-0.377193,-0.723473,2,10,0,-2,0,-1,4.021918,0,1,0,0,0,0,0,0,0


In [20]:
all_diff_cols = ["diff" + f for f in diff_features] + ["diff_dob"]

scaler = StandardScaler()
x_train[all_diff_cols] = scaler.fit_transform(x_train[all_diff_cols])
x_test[all_diff_cols] = scaler.transform(x_test[all_diff_cols])

joblib.dump(scaler, "../scalers/scaler_v1.joblib")

['../scalers/scaler_v1.joblib']

In [21]:
x_train.head()

,title_bout,diff_slpm,diff_str_acc,diff_sapm,diff_str_def,diff_td_avg,diff_td_acc,diff_td_def,diff_sub_avg,diff_height,diff_weight,diff_reach,diff_wins,diff_losses,diff_draws,diff_dob,Orthodox_vs_Orthodox,Orthodox_vs_Southpaw,Orthodox_vs_Switch,Southpaw_vs_Orthodox,Southpaw_vs_Southpaw,Southpaw_vs_Switch,Switch_vs_Orthodox,Switch_vs_Southpaw,Switch_vs_Switch
191,0,-1.215809,0.870950,1.314510,-3.752061,2.517988,-1.565794,-0.029466,0.040878,-0.365524,-1.895115,-0.610563,-0.257526,0.619472,0.020892,1.037777,1,0,0,0,0,0,0,0,0
260,0,-0.315207,-0.166635,0.138626,-0.341678,-0.205658,2.353074,-1.019981,0.040878,2.053290,-4.015872,0.628931,0.185088,0.294931,0.020892,-0.177421,1,0,0,0,0,0,0,0,0
274,1,-1.604364,-1.964544,-0.006623,-1.286996,1.284326,-0.706171,3.189706,-2.867237,1.247019,1.960805,0.009184,0.406395,0.619472,0.020892,0.725873,1,0,0,0,0,0,0,0,0
298,1,-0.868509,0.862034,-0.503587,0.757693,-1.248033,0.666773,-0.029466,-0.248453,-1.171795,0.032845,0.009184,-1.364062,-0.678692,0.020892,-0.709377,1,0,0,0,0,0,0,0,0
314,1,1.150611,2.756690,-0.655667,0.810773,2.587009,0.001754,-1.243715,-0.451730,0.843883,0.996825,0.009184,-0.478833,-0.029610,-1.956241,0.738644,0,1,0,0,0,0,0,0,0


In [22]:
len(x_train.columns)

25

In [23]:
x_test.head()

,title_bout,diff_slpm,diff_str_acc,diff_sapm,diff_str_def,diff_td_avg,diff_td_acc,diff_td_def,diff_sub_avg,diff_height,diff_weight,diff_reach,diff_wins,diff_losses,diff_draws,diff_dob,Orthodox_vs_Orthodox,Orthodox_vs_Southpaw,Orthodox_vs_Switch,Southpaw_vs_Orthodox,Southpaw_vs_Southpaw,Southpaw_vs_Switch,Switch_vs_Orthodox,Switch_vs_Southpaw,Switch_vs_Switch
7548,0,0.317579,-0.205453,-0.031527,0.025665,-1.159342,0.156009,-1.125909,0.334019,-0.768659,1.478815,0.009184,3.283389,3.540341,0.020892,-1.081206,0,0,0,0,1,0,0,0,0
7549,0,-0.220942,1.361064,-1.162034,-0.545860,0.383019,0.609899,-0.521284,-0.097672,-1.574930,0.032845,-0.920436,-0.700141,-1.003233,0.020892,1.701371,0,0,0,0,0,0,1,0,0
7550,0,-0.244397,1.438666,-0.939957,-0.980293,-0.487466,0.055807,2.116649,1.720944,-0.365524,-3.823076,-0.300689,-1.364062,-1.327774,-1.956241,0.080453,1,0,0,0,0,0,0,0,0
7551,0,-0.368841,-0.830735,-0.372703,-0.217451,-0.528981,0.191759,-0.396938,0.062718,0.440748,0.032845,0.009184,0.406395,-0.678692,1.998026,0.879615,0,0,1,0,0,0,0,0,0
7552,0,-2.201167,-0.508259,-1.096684,-1.192187,0.790667,-0.441249,-1.193967,0.154281,-0.365524,0.032845,-0.610563,1.955545,1.917636,0.020892,-1.802269,0,0,1,0,0,0,0,0,0


In [24]:
models = {
    "logistic_regression": LogisticRegression(),
    "knn": KNeighborsClassifier(n_neighbors=10)
}

predictions = {
    "logistic_regression": 0,
    "knn": 0
}

for model_name, model in models.items():
    models[model_name] = model.fit(x_train, y_train)
    predictions[model_name] = models[model_name].predict(x_test)

for model_name, prediction in predictions.items():
    print(f'{model_name} model accuracy: {accuracy_score(y_test, prediction)}')
    print(classification_report(y_test, prediction))


logistic_regression model accuracy: 0.6277372262773723
              precision    recall  f1-score   support

           0       0.65      0.64      0.65       363
           1       0.60      0.61      0.61       322

    accuracy                           0.63       685
   macro avg       0.63      0.63      0.63       685
weighted avg       0.63      0.63      0.63       685

knn model accuracy: 0.6014598540145986
              precision    recall  f1-score   support

           0       0.61      0.69      0.65       363
           1       0.59      0.50      0.54       322

    accuracy                           0.60       685
   macro avg       0.60      0.60      0.59       685
weighted avg       0.60      0.60      0.60       685



  File "c:\Users\marvi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\marvi\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\marvi\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\marvi\anaconda3\Lib\subprocess.

In [25]:
joblib.dump(models["logistic_regression"], "../models/log_reg_v1.joblib")

['../models/log_reg_v1.joblib']

In [26]:
joblib.dump(models["knn"], "../models/knn_v1.joblib")

['../models/knn_v1.joblib']